# Custom retriever and complex chains

- How t create a custom retriever, e.g. to get similarity score
- How to build a complex chai to score resume and output an evaluation of each resume

# Material

## Instance requirements
For AWS: Data Science 3.0 - Python 3.10 - ml.t3.medium - 2 vCPU + 4GB

## Initializations

In [ ]:
### Update environment

In [ ]:
!apt-get update && apt-get install -y build-essential 1>/dev/null

In [ ]:
!apt-get update && apt-get install -y jq 1>/dev/null

In [ ]:
!pip install --upgrade pip  1>/dev/null

## Requirements

In [ ]:
#!pip install langchain==0.0.230 1>/dev/null
!pip install langchain==0.0.266 1>/dev/null

In [ ]:
!pip install openai==0.27.8 1>/dev/null

In [ ]:
!pip install faiss-cpu==1.7.4 1>/dev/null

In [ ]:
!pip install tiktoken==0.4.0 1>/dev/null

### Jupyter extensions

## Secrets and credentials

In [ ]:
%%bash --out secrets 
# using AWS's Secret Manager to store keys
# garb the keys and store it into a Pytthon variable
export RESPONSE=$(aws secretsmanager get-secret-value --secret-id 'salvia/labbench/tests' )
export SECRETS=$( echo $RESPONSE | jq '.SecretString | fromjson')

echo $SECRETS

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = eval(secrets)["OPENAI_API_KEY"]


# Code session

In [ ]:
# common imports
from pprint import pprint
from typing import Any, Dict, List, Tuple

from langchain.schema import Document
from langchain.callbacks import get_openai_callback

# LLMs setup

## In memory Cache

In [ ]:
# cache reset
import langchain
from langchain.cache import InMemoryCache
langchain.llm_cache = InMemoryCache()

## LLM Setup

In [ ]:
from langchain.llms import OpenAI

llm = OpenAI(model_name="text-davinci-003")

In [ ]:
from langchain.chat_models import ChatOpenAI

chatllm = ChatOpenAI(model_name="gpt-3.5-turbo")

In [ ]:
#%pdef ChatOpenAI

In [ ]:
#%pinfo chatllm

In [ ]:
%%time
%%script echo skipping

with get_openai_callback() as cb:
    query = "What is the distance to the Moon?"
    response = chatllm.predict(query)
    print(f"{response=}")
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

# Load vector store db

Load a pre-built FAISS database with some resumes

In [ ]:
%%time

from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

with get_openai_callback() as cb:

    # setup an embedding model with same options
    embeddings = OpenAIEmbeddings(
        model="text-embedding-ada-002"
    )

    # load sample database
    index_name = "data/cv_index_faiss"
    sample_vector_store = FAISS.load_local(index_name, embeddings)
    
    print(f"database loaded.")

    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

In [ ]:
%pdef FAISS

In [ ]:
print(f"vector store has {len(sample_vector_store.docstore._dict)} items")

# Build components

## Step 1: Filter and score documents relevant for the query 

### Retrievers 

Vectore store has  search functions with score. These functions retrun pairs of (Document, floatt).

A retrieval system is defined as something that can take string queries and return
the most ‘relevant’ Documents from some source.

Built-in retriever ignore the score. They only return documents.

Will implement a retriever that calls a with_score function and keep track of the score as a metadata.

The metdata key is passed as a kwargs parameter.

Ressources:
- https://api.python.langchain.com/en/latest/schema/langchain.schema.retriever.BaseRetriever.html

#### investigations with search functions with score

In [ ]:
%%time
%%script echo skipping

with get_openai_callback() as cb:
    # check retrieval from vectore store
    k = 5
    score_threshold = 0.5  

    user_question = "candidate must have robotics skills"

    # may also have the max maxilul relevance with score
    results_with_scores = sample_vector_store.similarity_search_with_score(
        user_question,
        k=k,
        score_threshold=score_threshold
    )

    pprint(results_with_scores)
   
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


#### Custom retriever SimilaristScoreRetriever

TODO switch for similarity and MMR options

In [ ]:
from langchain.schema.retriever import BaseRetriever
from typing import List
from langchain.vectorstores.base import VectorStoreRetriever
from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from typing import Any, Dict, List, Tuple


class SimilarityScoreRetriever(VectorStoreRetriever):
    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        k = self.search_kwargs.get("k")
        score_threshold = self.search_kwargs.get("score_threshold")
        score_key = self.search_kwargs.get("score_key")
     
        results_with_scores = self.vectorstore.similarity_search_with_score(
            query,
            k=k,
            score_threshold=score_threshold
        )
        
        # return a list of paris (document, score)
        # put the score into the metadata of the dicument, under the key denoted by score_key
        def reformat(result: Tuple[Document, float]) -> Document:
            # set score 
            result[0].metadata[score_key] = result[1]
            # return only the documennt
            return result[0]
                                   
        documents = [reformat(result) for result in results_with_scores]
        
        return documents

In [ ]:
%%time
%%script echo skipping
# test custom retriever

with get_openai_callback() as cb:
    k = 5
    score_threshold = 0.5  
    search_type = "similarity_score_threshold"
    # toDO similarity without threshold
    score_key = "distance"
    search_kwargs = {"k": k, "score_threshold": score_threshold, "score_key": score_key}

    # lookup documents based on retrieveAr
    criteria = "candidate must have robotics skills"

    retriever = SimilarityScoreRetriever(
        vectorstore=sample_vector_store,
        search_type=search_type,
        search_kwargs=search_kwargs
    )

    results = retriever.get_relevant_documents(criteria)

    pprint(results)

    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


### Setup a source chain 

The source chain will use the retriever to get chunks of resules relevant to the query.

The chain's outputs are avaiable under the key "chunks" and because return_source_documents is True documents are avaiable under source_document.

e.g.
```
{'chunks': 'Based on the provided context, there is no explicit mention of '
           'robotics. However, the candidate should have a strong mastery of '
           'Python as it is mentioned multiple times in the technical '
           'environment section.',
 'query': 'the candidate must know robotics and master Python',
 'source_documents': [Document(page_content='Development of a matching tool for staffing offers and supply on availability. (Python, Power BI) Development of a tool to detect training paths compatible with the profiles of people on call.', metadata={'path': 'CV - Maria HONEURE - EN.pdf', 'type': 'application/pdf', 'distance': 0.4254169}),
...      
```

In [ ]:
# setup a simple memory 
from langchain.memory.simple import SimpleMemory

simple_memory = SimpleMemory()

In [ ]:
from langchain.chains import RetrievalQA

k = 5
score_threshold = 0.5  
search_type = "similarity_score_threshold"
# toDO similarity without threshold
score_key = "distance"
search_kwargs = {"k": k, "score_threshold": score_threshold, "score_key": score_key}

retriever = SimilarityScoreRetriever(
    vectorstore=sample_vector_store,
    search_type=search_type,
    search_kwargs=search_kwargs
)

#TODO prompt ?

source_chain = RetrievalQA.from_chain_type(
    llm=chatllm,
    chain_type="stuff", 
    retriever=retriever, 
    return_source_documents=True,
    output_key="source_documents",
    memory=simple_memory
    #chain_type_kwargs=chain_type_kwargs   # prompt
)
 

In [ ]:
%%time
#%%script echo skipping
# testing chain

criteria = "the candidate must know robotics and master Python"

with get_openai_callback() as cb:
    results = source_chain(criteria)
    
    pprint(results)
    
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

In [ ]:
from langchain.chains.sequential import SequentialChain
seq_chain = SequentialChain(
    chains=[source_chain], 
    input_variables=['query'],
    output_variables=['source_documents'],
    memory=simple_memory,
    verbose=True
)

In [ ]:
%%time
#%%script echo skipping
# testing chain

criteria = "the candidate must know robotics and master Python"

simple_memory = SimpleMemory()

with get_openai_callback() as cb:
    results = seq_chain(criteria)
    
    pprint(results)
    
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

## Step2 : Group by document name

The vector dataabse has mutiple chunks of resume for each canadidate.

A candidate may be idnetifyied by the filename.

The transform chain will group chunks of a chandidate in a document and compute a combined score

### groupby function

Transform passes the chain context as a dict of key values.

The context consists in output of previous chains and values stored in the memory. 

Must return a dict woth the output to store into the chain's context.

In [ ]:
from itertools import groupby

# will deal with this later
def compute_combined_score(group):
    return 1
    
def groupby_filename(inputs: dict) -> dict:
    chunks: List[Document] = inputs["source_documents"]

    chunks_sorted_by_filename = sorted(
        chunks, 
        key=lambda chunk: chunk.metadata['path'])
 
    groups: List[List[Document]] = []
    uniquekeys: List[str] = []
    for key, docs in groupby(chunks_sorted_by_filename, lambda chunk: chunk.metadata['path']):
        groups.append(list(docs))    # Store group iterator as a list
        uniquekeys.append(key)

    selections: List[Document] = []
    for key, group in zip(uniquekeys, groups):
        contents = [doc.page_content for doc in group]
        merged_content = "\n".join(contents)
        score =  compute_combined_score(group)
        doc = Document(
            page_content=merged_content, 
            metadata={
                'path': key, 
                'type': group[0].metadata['type'], 
                'score': score
            }
        )   
        selections.append(doc)
    
    return {"selections": selections} 

In [ ]:
%%time
%%script echo skipping
# testing groupby function

# make up a input ressembling the chain context
inputs = {
    'chunks': 'Whatever.',
    'query': 'Whatever',
    'source_documents': [
        Document(
            page_content='implementation of the Robotics Process Automation project', 
            metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                      'type': 'application/G_pdf', 
                      'distance': 0.41726395}
        ), Document(
            page_content='Robotics Process Automation projec : project scopingnd analysis', 
            metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                      'type': 'application/pdf', 
                      'distance': 0.42374027}
        ), Document(
            page_content='Data analysis with Python', 
            metadata={'path': 'CV - Patrick  B_20230203 - EN.pdf', 
                      'type': 'application/pdf', 
                      'distance': 0.45334827}
        )
    ]
}

result = groupby_filename(inputs)

pprint(result)

### combine score function

As we have scores for each chunks we want to add them up to compute a single score for each group of chunks.

Additicity will imply that a score is better if it is larger.

Please note that the score returned by similatity is a distance. 
A distance score is better if it is smaller.

In order to have values going in the same direction, 1/distance is added.
1/distance increase when distance decrease and can be added to each other safely.

Please note that it may fail if distance is 0? TODO


In [ ]:
def compute_combined_score(documents: List[Document]) -> float:
    # 1/x as x is good when near 0 but we sum
    # expects all documents to have a score metadata
    # and the score is > 0
    # tODO check score
    try:
        score =  sum([1/doc.metadata['distance']  for doc in documents]) 
    except Exception as e:
        raise Exception("missing distance or distance == 0 ")

    return score

In [ ]:
#%%time
#%%script echo skipping
# testing combone score function

# make up a group
documents = [
        Document(
            page_content='implementation of the Robotics Process Automation project', 
            metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                      'type': 'application/G_pdf', 
                      'distance': 0.41726395}
        ), Document(
            page_content='Robotics Process Automation projec : project scopingnd analysis', 
            metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                      'type': 'application/pdf', 
                      'distance': 0.42374027}
        )
]

score = compute_combined_score(documents)

print(score)


### transform chain

The transfor chain wraps a function which purpise is to transform the context.

It takes a list of keys to pass to the function, a function, a declares the list of outputs that will be added to the context.

- Input vaiables must exist in the context.
- Output variables must not exist and must match the dict returned by the function


In [ ]:
from langchain.chains import TransformChain

groupby_transform_chain = TransformChain(
    input_variables=["source_documents"], 
    output_variables=["selections"], 
    transform=groupby_filename,
    memory=simple_memory
)

In [ ]:
%%time
#%%script echo skipping
# testing transform chain 

# make up a input ressembling the chain context
source_documents = [
        Document(
            page_content='implementation of the Robotics Process Automation project', 
            metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                      'type': 'application/G_pdf', 
                      'distance': 0.41726395}
        ), Document(
            page_content='Robotics Process Automation projec : project scopingnd analysis', 
            metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                      'type': 'application/pdf', 
                      'distance': 0.42374027}
        ), Document(
            page_content='Data analysis with Python', 
            metadata={'path': 'CV - Patrick  B_20230203 - EN.pdf', 
                      'type': 'application/pdf', 
                      'distance': 0.45334827}
        )
    ]

with get_openai_callback() as cb:
    result = groupby_transform_chain(source_documents)
    pprint(result['selections'])
       
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

In [ ]:
from langchain.chains.sequential import SequentialChain
seq_chain = SequentialChain(
    chains=[groupby_transform_chain], 
    input_variables=['source_documents'],
    output_variables=['selections'],
    memory=simple_memory,
    verbose=True
)

### Testing a chain of steps 1 and 2 - TODO

In [ ]:
from langchain.chains.sequential import SequentialChain
seq_chain = SequentialChain(
    chains=[source_chain, groupby_transform_chain], 
    input_variables=['query'],
    output_variables=['selections'],
    memory=simple_memory,
    verbose=True
)

In [ ]:
%%time
%%script echo skipping
# testing step 1 and 2  

criteria = "the candidate must know robotics and master Python"

with get_openai_callback() as cb:
    result = seq_chain(criteria)
    pprint(result['selections'])
       
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

## Step 3: Summary & eval criteria 


https://api.python.langchain.com/en/latest/chains/langchain.chains.combine_documents.map_reduce.MapReduceDocumentsChain.html#langchain.chains.combine_documents.map_reduce.MapReduceDocumentsChain

In [ ]:
from langchain.chains import (
    StuffDocumentsChain,
    LLMChain,
    ReduceDocumentsChain,
    MapReduceDocumentsChain,
)
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI



In [ ]:
# This controls how each document will be formatted. Specifically,
# it will be passed to `format_document` - see that function for more
# details.
document_prompt = PromptTemplate(
    input_variables=["page_content", "metadata"],
    template="content:{page_content}\n{metadata}"
)
document_variable_name = "context"

# Mapper
#llm = OpenAI()
# The prompt here should take as an input variable the
# `document_variable_name`
#prompt = PromptTemplate.from_template(
#    "Summarize this content: {context}"
#)

prompt_template = """You are a HR assistant. 
Your job is to evaluate whether a candidate match the criteria.
You will be given a resume extarct.
Summarize the resume and explain why the candidate match the criteria.
Evaluate how the criteria and the resume match on a 3 level scale Low, Medium, High.
Add the path and the score to the response. 

Criteria: {query}
{context}  
Your Evaluation goes here
"""
prompt = PromptTemplate(
    template=prompt_template, 
    input_variables=["query",  "context"],    
)


mapper_chain = LLMChain(
    llm=chatllm, 
    prompt=prompt, 
    return_final_only=True)


In [ ]:
%%time
#%%script echo skipping
# testing EVALUATION

# make up a context
document = Document(
        page_content='implementation of the Robotics Process Automation project\nRobotics Process Automation projec : project scopingnd analysis', 
        metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                  'type': 'application/G_pdf', 
                  'score': 1.345
                 })

criteria = "the candidate must know robotics and master Python"

with get_openai_callback() as cb:
    result = mapper_chain({'query': criteria, 'context': document})
    pprint(result)
       
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
from langchain.chains import TransformChain
def concat_evaluations(inputs):
    evaluations = inputs['text']
    return "\n -*- \n".join(evaluations)

concat_transform_chain = TransformChain(
    input_variables=["text"], 
    output_variables=["report"], 
    transform=concat_evaluations,
    memory=simple_memory
)

In [ ]:

# We now define how to combine these summaries
#reduce_prompt = PromptTemplate.from_template(
#    "Combine these summaries: {context}"
#)
reduce_prompt = PromptTemplate.from_template(
    """
    Write the summary below. Do not alter content.
    
    summary {eval_context}
    """
)
reduce_llm_chain = LLMChain(llm=chatllm, prompt=reduce_prompt, return_final_only=True)

eval_document_prompt = PromptTemplate(
    input_variables=["page_content"],
    template="{page_content}"
)
eval_document_variable_name = "eval_context"

# combine_documents_chain is ALWAYS provided. 
# This is final chain that is called. 
# We pass all previous results to this chain, 
# and the output of this chain is returned as a final result.
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_llm_chain,
    document_prompt=eval_document_prompt,
    document_variable_name=eval_document_variable_name,
    document_separator="\n -*- \n"
)

# Combine documents by recursively reducing them.
# This involves combine_documents_chain or collapse_documents_chain
reduce_documents_chain = ReduceDocumentsChain(
    combine_documents_chain=combine_documents_chain
)

# Mapreduce chain
evaluation_chain = MapReduceDocumentsChain(
    llm_chain=mapper_chain,  
    document_variable_name=document_variable_name,
    reduce_documents_chain=reduce_documents_chain,
    memory=simple_memory,
    return_intermediate_steps=True
)

In [ ]:
%%time
#%%script echo skipping
# testing EVALUATION

# make up a context
selections = [
    Document(
        page_content='implementation of the Robotics Process Automation project\nRobotics Process Automation projec : project scopingnd analysis', 
        metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                  'type': 'application/G_pdf', 
                  'score': 1}),
    Document(
        page_content='Data analysis with Python', 
        metadata={'path': 'CV - Patrick  B_20230203 - EN.pdf', 
                  'type': 'application/pdf', 
                  'score': 1})
]

criteria = "the candidate must know robotics and master Python"

with get_openai_callback() as cb:
    result = evaluation_chain({'query': criteria, 'input_documents': selections})
    pprint(result)
       
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
## testtin chain for steo 1 and 2

## callback

In [ ]:
"""Callback Handler that prints to std out."""
from typing import Any, Dict, List, Optional, Union

from langchain.callbacks.base import BaseCallbackHandler
from langchain.schema import AgentAction, AgentFinish
from langchain.schema.messages import BaseMessage


from langchain.schema import LLMResult

from pathlib import Path
from datetime import datetime
import re
from uuid import UUID


class SnoopCallbackHandler(BaseCallbackHandler):
    """Callback Handler that snoops into the chain."""

    def __init__(self) -> None:
        """Initialize callback handler."""
        pass

    def on_llm_start(
        self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any
    ) -> None:
        """Print out the prompts."""
        print("\n=== on_llm_start ===")
        print("serialized")
        pprint(serialized)
        print("prompts")
        pprint(prompts)

    def on_chat_model_start(self, 
        serialized: Dict[str, Any], 
        messages: List[List[BaseMessage]], 
        run_id: UUID, parent_run_id: Optional[UUID] = None, 
        tags: Optional[List[str]] = None, 
        metadata: Optional[Dict[str, Any]] = None, 
        **kwargs: Any
    ) -> Any:
        print("\n=== on_chain_start ===")

    def on_llm_end(self, response: LLMResult, **kwargs: Any) -> None:
        """Do nothing."""
        print("\n=== on_llm_end ===")
        print("response")
        pprint(response)

    def on_chain_start(
        self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any
    ) -> None:
        """Print out that we are entering a chain."""
        print("\n=== on_chain_start ===")
        print("serialized")
        pprint(serialized)
        print("inputs")
        pprint(inputs)

    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any) -> None:
        """Print out that we finished a chain."""
        print("\n=== on_chain_end ===")
        print("outputs")
        pprint(outputs)

   # def save_evaluation_as_file(self):

        #target_file = f"{self.path.stem}.html" 
        #with open(target_file, "w") as f:
        #    f.write()
        #print(f"Saved  content to {target_file}")



In [ ]:
snooped_eval_chain = LLMChain(
    llm=chatllm, 
    prompt=prompt, 
    callbacks=[SnoopCallbackHandler()], 
    return_final_only=True)


In [ ]:
%%time
#%%script echo skipping
# testing EVALUATION

# make up a context
documents = [
    Document(
        page_content='implementation of the Robotics Process Automation project', 
        metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                  'type': 'application/G_pdf', 
                  'score': 1.345
                 }),
    Document(
        page_content='project scopingnd analysis', 
        metadata={'path': 'CV - Patrick B_20230203 - EN.pdf', 
                  'type': 'application/G_pdf', 
                  'score': 2.104
                 })
]

criteria = "the candidate must know robotics and master Python"

with get_openai_callback() as cb:
    result = snooped_eval_chain({'query': criteria, 'context': documents[0]})
    pprint(result)
       
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
import json
import re

class EnhanceEvaluationCallbackHandler(BaseCallbackHandler):
    """Callback Handler that snoops into the chain."""

    def __init__(self, memory) -> None:
        """Initialize callback handler."""
        self.memory = memory
        self.input_document = None
        self.regex_name = re.compile("^CV - ([^-_]*)_[0-9]*")
 
    def on_chain_start(
        self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any
    ) -> None:
        """Print out that we are entering a chain."""
        self.input_document = inputs.get('context')

        # TODO on apply multi documeents

    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any) -> None:
        """Print out that we finished a chain."""
        text = outputs.get('text')

        try:
            file_name = self.input_document.metadata.get('path')
            match = self.regex_name.search(file_name)
            if match:
                name = match.group(1)
            else:
                name = "UNDEFINED"
        except Exception as e:
            name = "UNDEFINED"

        final_document = Document(
            page_content=text, 
            metadata={'path': self.input_document.metadata.get('path'), 
                      'type': self.input_document.metadata.get('type'), 
                      'score': self.input_document.metadata.get('score'), 
                      'fragments': self.input_document.page_content,
                      'name': name
                     })
        self.memory.memories['final_document'] = final_document

        json_dict = {
            'name': final_document.metadata['name'],
            'evaluation': final_document.page_content,
            'fragments': final_document.metadata['fragments'],
            'source_path': final_document.metadata['path'],
            'source_file_type': final_document.metadata['type'],
            'combined_score': final_document.metadata['score'],   
        }
        json_string = json.dumps(json_dict)
        self.memory.memories['json_output'] = json_string


In [ ]:
simple_memory = SimpleMemory()

enhanced_eval_chain = LLMChain(
    llm=chatllm, 
    prompt=prompt, 
    memory=simple_memory,
    callbacks=[EnhanceEvaluationCallbackHandler(simple_memory)], 
    return_final_only=True)


In [ ]:
%%time
#%%script echo skipping
# testing EVALUATION

# make up a context
documents = [
    Document(
        page_content='implementation of the Robotics Process Automation project', 
        metadata={'path': 'CV - Daniel G_20230203 - EN.pdf', 
                  'type': 'application/G_pdf', 
                  'score': 1.345
                 }),
    Document(
        page_content='project scopingnd analysis', 
        metadata={'path': 'CV - Patrick B_20230203 - EN.pdf', 
                  'type': 'application/G_pdf', 
                  'score': 2.104
                 })
]

criteria = "the candidate must know robotics and master Python"

with get_openai_callback() as cb:
    result = enhanced_eval_chain({'query': criteria, 'context': documents[0]})
    print('result')
    pprint(result)
    print('memory')
    pprint(simple_memory.memories)
       
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


# Sequential chain

In [ ]:
TODO map reduce chain avec un reduce vide

# ???

In [ ]:
# If we wanted to, we could also pass in collapse_documents_chain
# which is specifically aimed at collapsing documents BEFORE
# the final call.
prompt = PromptTemplate.from_template(
    "Collapse this content: {context}"
)
llm_chain = LLMChain(llm=llm, prompt=prompt)
collapse_documents_chain = StuffDocumentsChain(
    llm_chain=llm_chain,
    document_prompt=document_prompt,
    document_variable_name=document_variable_name
)
reduce_documents_chain = ReduceDocumentsChain(
    combine_documents_chain=combine_documents_chain,
    collapse_documents_chain=collapse_documents_chain,
)
chain = MapReduceDocumentsChain(
    llm_chain=llm_chain,
    reduce_documents_chain=reduce_documents_chain,
)

In [ ]:
from langchain.prompts import PromptTemplate

# LLM chain consisting of the LLM and a prompt
prompt_template = """You are a HR assistant. 
Your job is to evaluate whether a candidate match the criteria.
You will be given a resume extarcts.
Summarize the resume and explain why the candidate match the criteria.
Evaluate how the criteria and the resume match on a 2 level scale Low, Medium, High.

Criteria: {query}
Resume extract: {context}  
Your Evaluation goes here
"""
prompt = PromptTemplate(
    template=prompt_template, 
    input_variables=["query",  "context"],    
)

In [ ]:
'''
from langchain import FewShotPromptTemplate

# create our examples
examples = [
    {
        "query": "How are you?",
        "answer": "I can't complain but sometimes I still do."
    }, {
        "query": "What time is it?",
        "answer": "It's time to get a watch."
    }
]

# create a example template
example_template = """
User: {query}
AI: {answer}
"""

# create a prompt example from above template
example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template=example_template
)

# now break our previous prompt into a prefix and suffix
# the prefix is our instructions
prefix = """The following are exerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative  and funny responses to the users questions. Here are some
examples: 
"""
# and the suffix our user input and output indicator
suffix = """
User: {query}
AI: """

# now create the few shot prompt template
few_shot_prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["query"],
    example_separator="\n\n"
)
'''

# Pragmatic process

In [ ]:
from pprint import pprint
from typing import Any, Dict, List, Tuple

from langchain.schema import Document
from langchain.callbacks import get_openai_callback

In [ ]:
# cache reset
import langchain
from langchain.cache import InMemoryCache
langchain.llm_cache = InMemoryCache()

In [ ]:
from langchain.chat_models import ChatOpenAI

chatllm = ChatOpenAI(model_name="gpt-3.5-turbo")

In [ ]:
# setup a simple memory 
from langchain.memory.simple import SimpleMemory

simple_memory = SimpleMemory()

In [ ]:
%%time
# load a prebiult vector store

from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

with get_openai_callback() as cb:

    # setup an embedding model with same options
    embeddings = OpenAIEmbeddings(
        model="text-embedding-ada-002"
    )

    # load sample database
    index_name = "data/cv_index_faiss"
    sample_vector_store = FAISS.load_local(index_name, embeddings)
    
    print(f"database loaded.")
    print(f"vector store has {len(sample_vector_store.docstore._dict)} items")

    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

In [ ]:
from langchain.schema.retriever import BaseRetriever
from typing import List
from langchain.vectorstores.base import VectorStoreRetriever
from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from typing import Any, Dict, List, Tuple


class SimilarityScoreRetriever(VectorStoreRetriever):
    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        k = self.search_kwargs.get("k")
        score_threshold = self.search_kwargs.get("score_threshold")
        score_key = self.search_kwargs.get("score_key")
     
        results_with_scores = self.vectorstore.similarity_search_with_score(
            query,
            k=k,
            score_threshold=score_threshold
        )
        
        # return a list of paris (document, score)
        # put the score into the metadata of the dicument, under the key denoted by score_key
        def reformat(result: Tuple[Document, float]) -> Document:
            # set score 
            result[0].metadata[score_key] = result[1]
            # return only the documennt
            return result[0]
                                   
        documents = [reformat(result) for result in results_with_scores]
        
        return documents

In [ ]:
from langchain.chains import RetrievalQA

k = 5
score_threshold = 0.5  
search_type = "similarity_score_threshold"
# toDO similarity without threshold
score_key = "distance"
search_kwargs = {"k": k, "score_threshold": score_threshold, "score_key": score_key}

retriever = SimilarityScoreRetriever(
    vectorstore=sample_vector_store,
    search_type=search_type,
    search_kwargs=search_kwargs
)

#TODO prompt ?

source_chain = RetrievalQA.from_chain_type(
    llm=chatllm,
    chain_type="stuff", 
    retriever=retriever, 
    return_source_documents=True,
    output_key="source_documents",
    memory=simple_memory
    #chain_type_kwargs=chain_type_kwargs   # prompt
)
 

In [ ]:
%%time
#%%script echo skipping
# testing chain

criteria = "the candidate must know robotics and master Python"

with get_openai_callback() as cb:
    results = source_chain(criteria)
    
    pprint(results)
    
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

In [ ]:
source_documents = results.get('source_documents')
                             
print(len(source_documents))

In [ ]:
from itertools import groupby

def compute_combined_score(documents: List[Document]) -> float:
    # 1/x as x is good when near 0 but we sum
    # expects all documents to have a score metadata
    # and the score is > 0
    # tODO check score
    try:
        score =  sum([1/doc.metadata['distance']  for doc in documents]) 
    except Exception as e:
        raise Exception("missing distance or distance == 0 ")

    return score
    
def groupby_filename(inputs: dict) -> dict:
    chunks: List[Document] = inputs["source_documents"]

    chunks_sorted_by_filename = sorted(
        chunks, 
        key=lambda chunk: chunk.metadata['path'])
 
    groups: List[List[Document]] = []
    uniquekeys: List[str] = []
    for key, docs in groupby(chunks_sorted_by_filename, lambda chunk: chunk.metadata['path']):
        groups.append(list(docs))    # Store group iterator as a list
        uniquekeys.append(key)

    selections: List[Document] = []
    for key, group in zip(uniquekeys, groups):
        contents = [doc.page_content for doc in group]
        merged_content = "\n".join(contents)
        score =  compute_combined_score(group)
        doc = Document(
            page_content=merged_content, 
            metadata={
                'path': key, 
                'type': group[0].metadata['type'], 
                'score': score
            }
        )   
        selections.append(doc)
    
    return {"selections": selections} 

In [ ]:
from langchain.chains import TransformChain

groupby_transform_chain = TransformChain(
    input_variables=["source_documents"], 
    output_variables=["selections"], 
    transform=groupby_filename,
    memory=simple_memory
)

In [ ]:
%%time
#%%script echo skipping
# testing transform chain 

with get_openai_callback() as cb:
    results = groupby_transform_chain(source_documents)
    pprint(result['selections'])
       
    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")

In [ ]:
merged_documents = results.get('selections')

print(len(merged_documents))

In [ ]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

prompt_template = """You are a HR assistant. 
Your job is to evaluate whether a candidate match the criteria.
You will be given a resume extarct.
Summarize the resume and explain why the candidate match the criteria.
Evaluate how the criteria and the resume match on a 3 level scale Low, Medium, High.
Add the path and the score to the response. 

Criteria: {query}
{context}  
Your Evaluation goes here
"""
prompt = PromptTemplate(
    template=prompt_template, 
    input_variables=["query",  "context"],    
)


evalution_chain = LLMChain(
    llm=chatllm, 
    prompt=prompt, 
    memory=simple_memory,
    return_final_only=True)


In [ ]:
#%%script echo skipping
# testing EVALUATION

evaluations = []
with get_openai_callback() as cb:
    for document in merged_documents:
        result = evalution_chain({'query': criteria, 'context': document})
        pprint(result)
        evaluations.append(result.get('text'))

    print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")


In [ ]:
print(len(evaluations))
pprint(evaluations)


In [ ]:
%%time
#%%script echo skipping
# testing chain

criteria = "the candidate must know robotics and master Python"


def resume_evaluation_process(criteria: str):
    with get_openai_callback() as cb:
        results = source_chain(criteria)
        source_documents = results.get('source_documents')
        
        results = groupby_transform_chain(source_documents)
        merged_documents = results.get('selections')

        evaluations = []
        for document in merged_documents:
            result = evalution_chain({'query': criteria, 'context': document})
            evaluation_document = Document(      
                page_content=result.get('text'),
                metadata={'path':  document.metadata.get('path'), 
                          'type': document.metadata.get('type'), 
                          'score': document.metadata.get('score'), 
                         })

            evaluations.append(evaluation_document)

        return (evaluations, cb)
        
evaluations, cb = resume_evaluation_process(criteria)
for evaluation in evaluations:
    pprint(evaluation)    
    print("\n")
    
print(f"\nUsage monitoring: token used={cb.total_tokens}, requests={cb.successful_requests}, total cost (USD)={cb.total_cost} \n")
